# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR$^2$](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# `metadata` is an object with attribute access
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
List all available record sets and their fields using their `@id` references.

In [ ]:
from pprint import pprint

# List all record sets by @id and name
print("Available record sets (by @id and name):\n")
record_sets = dataset.metadata.recordSet
record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', str(rs))
    name = getattr(rs, 'name', 'N/A')
    print(f"- @id: {rs_id}    name: {name}")
    record_set_ids.append(rs_id)

print("\nFields per record set (@id, name):\n")
record_set_fields = {}
for rs in record_sets:
    rs_id = getattr(rs, '@id', str(rs))
    fields = getattr(rs, 'field', [])
    record_set_fields[rs_id] = []
    print(f"RecordSet {rs_id}:")
    for field in fields:
        field_id = getattr(field, '@id', str(field))
        field_name = getattr(field, 'name', 'N/A')
        print(f"    - Field @id: {field_id}   name: {field_name}")
        record_set_fields[rs_id].append(field_id)
    print()

# Show the mapping for reference
print("Summary of record_set_ids:")
pprint(record_set_ids)


## 3. Data Extraction
Load each record set as a pandas DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Prepare to load each record set
dataframes = {}
print(f"Extracting {len(record_set_ids)} record sets to pandas DataFrames...")
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id={record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Shape: {df.shape}\nSample columns: {df.columns.tolist()}")
            print(df.head(2))
        else:
            print("  [No records loaded]")
    except Exception as e:
        print(f"  [Error: {e}]")
    print()
# For this notebook, we will pick the first record set with data
for rs_id in record_set_ids:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break
print(f"We'll use main_record_set_id: {main_record_set_id}")
print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (filtering, normalization, grouping) referencing fields by their `@id` when possible.

In [ ]:
# Try to identify a numeric field (by checking dtypes)
df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include=["number"]).columns.tolist()
print(f"Numeric fields detected: {numeric_fields}")

if len(numeric_fields) > 0:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field for EDA: {numeric_field}")
    threshold = df[numeric_field].mean()  # Use mean as example threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (count={len(filtered_df)}):\n")
    print(filtered_df.head())

    # Normalizing
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field}:\n",
          filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print("No numeric field detected for EDA.")

# Try grouping by a non-numeric (categorical) field
cat_fields = [c for c in df.columns if df[c].dtype == object]
if len(cat_fields) > 0 and len(numeric_fields) > 0:
    group_field = cat_fields[0]
    print(f"\nGrouping by categorical field: {group_field} (mean of {numeric_field})")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped results:\n{grouped_df.head()}")

## 5. Visualization
Visualize the distribution of the chosen numeric field or the relationship between two fields if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes[main_record_set_id]) and len(numeric_fields) > 0:
    numeric_field = numeric_fields[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have a categorical field, plot mean by group
    if len(cat_fields) > 0:
        group_field = cat_fields[0]
        means = df.groupby(group_field)[numeric_field].mean().dropna()
        means.plot(kind="bar", figsize=(10,4), title=f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
In this notebook, we have used `mlcroissant` to load, overview, and explore the FAIR$^2$ dataset. We extracted record sets using their `@id`, explored fields, performed simple EDA including filtering, normalization, grouping and visualization of numeric data, all in a reproducible manner using Croissant's standard schema representation.

**Next steps** might include advanced analysis, joining record sets, or feature engineering tailored to downstream scientific or machine learning applications.